#### Load variables

In [44]:
date_str_1 = "2025-11-01"
date_str_2 = "2025-11-15"
date_str_3 = "2025-11-29"

current_snapshot_date_str = date_str_3

In [45]:
# =========================
# Load outputs from build_snapshots notebook
# =========================
import pandas as pd

student_snapshot = pd.read_csv(f"ProcessedData/student_snapshot_{current_snapshot_date_str}.csv")

low_performance_students = pd.read_csv(f"ProcessedData/low_performance_students_{current_snapshot_date_str}.csv")
declining_performance_students = pd.read_csv(f"ProcessedData/declining_performance_students_{current_snapshot_date_str}.csv")
low_submission_students = pd.read_csv(f"ProcessedData/low_submission_students_{current_snapshot_date_str}.csv")
top_students = pd.read_csv(f"ProcessedData/top_students_{current_snapshot_date_str}.csv")

print("Loaded all prepared data.")

Loaded all prepared data.


#### LLM

Low Performance

In [46]:
# =========================
# Low performance LLM analysis + parsing + memory
# =========================
import json
import re
from llm_client import llm_call

def safe_parse_llm_json(text):
    try:
        return json.loads(text)
    except:
        pass

    try:
        json_match = re.search(r"\[.*\]", text, re.DOTALL)
        if json_match:
            return json.loads(json_match.group())
    except:
        pass

    print("⚠️ Failed to parse LLM output as JSON")
    return None

# -------------------------
# Prepare data for LLM
# -------------------------
low_performance_for_llm = low_performance_students[
    [
        "FirstName",
        "LastName",
        "average_grade",
        "recent_average_grade",
        "submission_rate",
        "late_submissions",
        "days_since_last_submission",
    ]
].copy()

student_table_text = low_performance_for_llm.to_string(index=False)

# -------------------------
# Prompt
# -------------------------
system_prompt = """
You are an academic support assistant for a homeroom teacher.

You will receive a small table of students who were already identified by code as low performers.
Your task is NOT to find new students.
Your task is ONLY to classify the type of problem for each student.

Possible labels:
1. struggling_academically
   Use this when the student submits most assignments, but grades are very low.
   This suggests effort is present, but the student may not understand the material well.

2. disengaged
   Use this when the student shows lower engagement patterns, such as lower submission rate,
   many late submissions, or weak participation together with poor grades.

3. unclear
   Use this when the pattern is mixed and there is not enough evidence to confidently choose one of the two labels above.

Important rules:
- Be cautious.
- Do not call a student lazy or stupid.
- Use only the provided data.
- Return only valid JSON.
- For each student include:
  first_name, last_name, label, short_reason
"""

user_prompt = f"""
Student data:
{student_table_text}

Return JSON in this exact format:
[
  {{
    "first_name": "Name",
    "last_name": "Surname",
    "label": "struggling_academically",
    "short_reason": "High submission rate but extremely low grades."
  }}
]
"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]

# -------------------------
# Run LLM
# -------------------------
low_performance_llm_output = llm_call(
    messages,
    max_new_tokens=400
)

print("Raw LLM output:")
print(low_performance_llm_output)

# -------------------------
# Parse
# -------------------------
parsed_low_performance = safe_parse_llm_json(low_performance_llm_output)

print("\nParsed LLM output:")
print(parsed_low_performance)

# -------------------------
# Build memory with IDs
# -------------------------
memory_students = []

if parsed_low_performance is not None:
    for item in parsed_low_performance:
        first_name = item.get("first_name")
        last_name = item.get("last_name")

        match = student_snapshot[
            (student_snapshot["FirstName"] == first_name) &
            (student_snapshot["LastName"] == last_name)
        ]

        if len(match) == 0:
            print(f"⚠️ Student not found: {first_name} {last_name}")
            continue

        student_id = int(match.iloc[0]["student_id"])

        memory_students.append({
            "student_id": student_id,
            "first_name": first_name,
            "last_name": last_name,
            "label": item.get("label"),
            "reason": item.get("short_reason"),
            "last_updated": "2025-11-01",
            "signal_type": "low_performance"
        })

print("\nMemory entries:", len(memory_students))
print(memory_students[:5])

# -------------------------
# Save memory
# -------------------------
with open(f"Memory/memory_weak_students_{current_snapshot_date_str}.json", "w", encoding="utf-8") as f:
    json.dump(memory_students, f, ensure_ascii=False, indent=2)

print("\nMemory saved to memory_weak_students.json")

Raw LLM output:
[
  {
    "first_name": "Itay",
    "last_name": "Mizrahi",
    "label": "struggling_academically",
    "short_reason": "High submission rate with very low average grades."
  },
  {
    "first_name": "Dan",
    "last_name": "Cohen",
    "label": "struggling_academically",
    "short_reason": "High submission rate with low average grades."
  },
  {
    "first_name": "Noa",
    "last_name": "Levy",
    "label": "struggling_academically",
    "short_reason": "High submission rate with low average grades."
  }
]

Parsed LLM output:
[{'first_name': 'Itay', 'last_name': 'Mizrahi', 'label': 'struggling_academically', 'short_reason': 'High submission rate with very low average grades.'}, {'first_name': 'Dan', 'last_name': 'Cohen', 'label': 'struggling_academically', 'short_reason': 'High submission rate with low average grades.'}, {'first_name': 'Noa', 'last_name': 'Levy', 'label': 'struggling_academically', 'short_reason': 'High submission rate with low average grades.'}]

Mem

Declining performance

In [47]:
# =========================
# Declining performance LLM analysis + parsing + memory
# =========================
import json
import re
from llm_client import llm_call

def safe_parse_llm_json(text):
    try:
        return json.loads(text)
    except:
        pass

    try:
        json_match = re.search(r"\[.*\]", text, re.DOTALL)
        if json_match:
            return json.loads(json_match.group())
    except:
        pass

    print("⚠️ Failed to parse LLM output as JSON")
    return None

# -------------------------
# Prepare data for LLM
# -------------------------
declining_for_llm = declining_performance_students[
    [
        "FirstName",
        "LastName",
        "average_grade",
        "recent_average_grade",
        "grade_drop",
        "submission_rate",
        "late_submissions",
        "days_since_last_submission",
    ]
].copy()

student_table_text = declining_for_llm.to_string(index=False)

# -------------------------
# Prompt
# -------------------------
system_prompt = """
You are an academic support assistant for a homeroom teacher.

You will receive a small table of students who were already identified by code as students with declining performance.
Your task is NOT to find new students.
Your task is ONLY to classify the type of decline for each student.

Possible labels:
1. temporary_drop
   Use this when the student's recent results are worse than usual, but the overall pattern still looks mostly stable.
   This suggests a temporary setback or short-term fluctuation.

2. worsening_trend
   Use this when the recent decline looks serious enough to suggest that the student may be entering a broader downward trend.

3. unclear
   Use this when the pattern is mixed and there is not enough evidence to confidently choose one of the two labels above.

Important rules:
- Be cautious.
- Use only the provided data.
- Do not invent hidden causes.
- Return only valid JSON.
- For each student include:
  first_name, last_name, label, short_reason
"""

user_prompt = f"""
Student data:
{student_table_text}

Return JSON in this exact format:
[
  {{
    "first_name": "Name",
    "last_name": "Surname",
    "label": "worsening_trend",
    "short_reason": "Recent average is much lower than overall average, suggesting a meaningful decline."
  }}
]
"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]

# -------------------------
# Run LLM
# -------------------------
declining_llm_output = llm_call(
    messages,
    max_new_tokens=500
)

print("Raw LLM output:")
print(declining_llm_output)

# -------------------------
# Parse
# -------------------------
parsed_declining = safe_parse_llm_json(declining_llm_output)

print("\nParsed LLM output:")
print(parsed_declining)

# -------------------------
# Build memory with IDs
# -------------------------
memory_declining = []

if parsed_declining is not None:
    for item in parsed_declining:
        first_name = item.get("first_name")
        last_name = item.get("last_name")

        match = student_snapshot[
            (student_snapshot["FirstName"] == first_name) &
            (student_snapshot["LastName"] == last_name)
        ]

        if len(match) == 0:
            print(f"⚠️ Student not found: {first_name} {last_name}")
            continue

        student_id = int(match.iloc[0]["student_id"])

        memory_declining.append({
            "student_id": student_id,
            "first_name": first_name,
            "last_name": last_name,
            "signal_type": "declining_performance",
            "label": item.get("label"),
            "reason": item.get("short_reason"),
            "last_updated": "2025-11-01"
        })

print("\nDeclining memory entries:", len(memory_declining))
print(memory_declining[:5])

# -------------------------
# Save memory
# -------------------------
with open(f"Memory/memory_declining_{current_snapshot_date_str}.json", "w", encoding="utf-8") as f:
    json.dump(memory_declining, f, ensure_ascii=False, indent=2)

print("\nMemory saved to memory_declining.json")

Raw LLM output:
[
  {
    "first_name": "Rotem",
    "last_name": "Golan",
    "label": "worsening_trend",
    "short_reason": "Recent average grade dropped significantly from overall average, indicating a serious decline."
  }
]

Parsed LLM output:
[{'first_name': 'Rotem', 'last_name': 'Golan', 'label': 'worsening_trend', 'short_reason': 'Recent average grade dropped significantly from overall average, indicating a serious decline.'}]

Declining memory entries: 1
[{'student_id': 29, 'first_name': 'Rotem', 'last_name': 'Golan', 'signal_type': 'declining_performance', 'label': 'worsening_trend', 'reason': 'Recent average grade dropped significantly from overall average, indicating a serious decline.', 'last_updated': '2025-11-01'}]

Memory saved to memory_declining.json


Low submission

In [48]:
# =========================
# Low submission LLM analysis + parsing + memory
# =========================
import json
import re
from llm_client import llm_call

def safe_parse_llm_json(text):
    try:
        return json.loads(text)
    except:
        pass

    try:
        json_match = re.search(r"\[.*\]", text, re.DOTALL)
        if json_match:
            return json.loads(json_match.group())
    except:
        pass

    print("⚠️ Failed to parse LLM output as JSON")
    return None

# -------------------------
# Prepare data for LLM
# -------------------------
low_submission_for_llm = low_submission_students[
    [
        "FirstName",
        "LastName",
        "submitted_assignments",
        "total_assignments_available",
        "submission_rate",
        "late_submissions",
        "average_grade",
        "recent_average_grade",
        "days_since_last_submission",
    ]
].copy()

student_table_text = low_submission_for_llm.to_string(index=False)

# -------------------------
# Prompt
# -------------------------
system_prompt = """
You are an academic support assistant for a homeroom teacher.

You will receive a small table of students who were already identified by code as students with low homework submission.
Your task is NOT to find new students.
Your task is ONLY to classify the level/type of low engagement for each student.

Possible labels:
1. mild_low_engagement
   Use this when the student misses some assignments, but the pattern is not severe.
   This suggests a moderate engagement issue.

2. serious_low_engagement
   Use this when the student shows a clearly concerning engagement pattern,
   such as notably reduced submission rate, many late submissions, or weak participation signals.

3. unclear
   Use this when the pattern is mixed and there is not enough evidence to confidently choose one of the two labels above.

Important rules:
- Be cautious.
- Use only the provided data.
- Do not invent hidden causes.
- Return only valid JSON.
- For each student include:
  first_name, last_name, label, short_reason
"""

user_prompt = f"""
Student data:
{student_table_text}

Return JSON in this exact format:
[
  {{
    "first_name": "Name",
    "last_name": "Surname",
    "label": "serious_low_engagement",
    "short_reason": "Submission rate is low and the student also has multiple late submissions."
  }}
]
"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]

# -------------------------
# Run LLM
# -------------------------
low_submission_llm_output = llm_call(
    messages,
    max_new_tokens=500
)

print("Raw LLM output:")
print(low_submission_llm_output)

# -------------------------
# Parse
# -------------------------
parsed_low_submission = safe_parse_llm_json(low_submission_llm_output)

print("\nParsed LLM output:")
print(parsed_low_submission)

# -------------------------
# Build memory with IDs
# -------------------------
memory_low_submission = []

if parsed_low_submission is not None:
    for item in parsed_low_submission:
        first_name = item.get("first_name")
        last_name = item.get("last_name")

        match = student_snapshot[
            (student_snapshot["FirstName"] == first_name) &
            (student_snapshot["LastName"] == last_name)
        ]

        if len(match) == 0:
            print(f"⚠️ Student not found: {first_name} {last_name}")
            continue

        student_id = int(match.iloc[0]["student_id"])

        memory_low_submission.append({
            "student_id": student_id,
            "first_name": first_name,
            "last_name": last_name,
            "signal_type": "low_submission",
            "label": item.get("label"),
            "reason": item.get("short_reason"),
            "last_updated": "2025-11-01"
        })

print("\nLow submission memory entries:", len(memory_low_submission))
print(memory_low_submission[:5])

# -------------------------
# Save memory
# -------------------------
with open(f"Memory/memory_low_submission_{current_snapshot_date_str}.json", "w", encoding="utf-8") as f:
    json.dump(memory_low_submission, f, ensure_ascii=False, indent=2)

print("\nMemory saved to memory_low_submission.json")

Raw LLM output:
[]

Parsed LLM output:
[]

Low submission memory entries: 0
[]

Memory saved to memory_low_submission.json


Top students

In [49]:
# =========================
# Top students LLM analysis + parsing + memory
# =========================
import json
import re
from llm_client import llm_call

def safe_parse_llm_json(text):
    try:
        return json.loads(text)
    except:
        pass

    try:
        json_match = re.search(r"\[.*\]", text, re.DOTALL)
        if json_match:
            return json.loads(json_match.group())
    except:
        pass

    print("⚠️ Failed to parse LLM output as JSON")
    return None

# -------------------------
# Prepare data for LLM
# -------------------------
top_students_for_llm = top_students[
    [
        "FirstName",
        "LastName",
        "average_grade",
        "recent_average_grade",
        "submission_rate",
        "late_submissions",
        "days_since_last_submission",
    ]
].copy()

student_table_text = top_students_for_llm.to_string(index=False)

# -------------------------
# Prompt
# -------------------------
system_prompt = """
You are an academic support assistant for a homeroom teacher.

You will receive a small table of students who were already identified by code as top students.
Your task is NOT to find new students.
Your task is ONLY to classify the type of strong performance for each student.

Possible labels:
1. consistently_excellent
   Use this when the student shows strong overall results and the pattern looks stable.

2. excellent_with_positive_trend
   Use this when the student is strong overall and the recent results suggest additional positive momentum or improvement.

3. unclear
   Use this when the pattern is mixed and there is not enough evidence to confidently choose one of the two labels above.

Important rules:
- Be cautious.
- Use only the provided data.
- Do not invent hidden causes.
- Return only valid JSON.
- For each student include:
  first_name, last_name, label, short_reason
"""

user_prompt = f"""
Student data:
{student_table_text}

Return JSON in this exact format:
[
  {{
    "first_name": "Name",
    "last_name": "Surname",
    "label": "consistently_excellent",
    "short_reason": "High overall grades with stable strong recent performance."
  }}
]
"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]

# -------------------------
# Run LLM
# -------------------------
top_students_llm_output = llm_call(
    messages,
    max_new_tokens=500
)

print("Raw LLM output:")
print(top_students_llm_output)

# -------------------------
# Parse
# -------------------------
parsed_top_students = safe_parse_llm_json(top_students_llm_output)

print("\nParsed LLM output:")
print(parsed_top_students)

# -------------------------
# Build memory with IDs
# -------------------------
memory_top_students = []

if parsed_top_students is not None:
    for item in parsed_top_students:
        first_name = item.get("first_name")
        last_name = item.get("last_name")

        match = student_snapshot[
            (student_snapshot["FirstName"] == first_name) &
            (student_snapshot["LastName"] == last_name)
        ]

        if len(match) == 0:
            print(f"⚠️ Student not found: {first_name} {last_name}")
            continue

        student_id = int(match.iloc[0]["student_id"])

        memory_top_students.append({
            "student_id": student_id,
            "first_name": first_name,
            "last_name": last_name,
            "signal_type": "top_students",
            "label": item.get("label"),
            "reason": item.get("short_reason"),
            "last_updated": "2025-11-01"
        })

print("\nTop students memory entries:", len(memory_top_students))
print(memory_top_students[:5])

# -------------------------
# Save memory
# -------------------------
with open(f"Memory/memory_top_students_{current_snapshot_date_str}.json", "w", encoding="utf-8") as f:
    json.dump(memory_top_students, f, ensure_ascii=False, indent=2)

print("\nMemory saved to memory_top_students.json")

Raw LLM output:
[
  {
    "first_name": "Sivan",
    "last_name": "Elkayam",
    "label": "consistently_excellent",
    "short_reason": "High overall grades with stable strong recent performance."
  },
  {
    "first_name": "Or",
    "last_name": "Turgeman",
    "label": "excellent_with_positive_trend",
    "short_reason": "Strong overall grades with recent improvement in average grade."
  },
  {
    "first_name": "Nir",
    "last_name": "Hazan",
    "label": "excellent_with_positive_trend",
    "short_reason": "Good overall grades with a notable recent increase in average grade."
  }
]

Parsed LLM output:
[{'first_name': 'Sivan', 'last_name': 'Elkayam', 'label': 'consistently_excellent', 'short_reason': 'High overall grades with stable strong recent performance.'}, {'first_name': 'Or', 'last_name': 'Turgeman', 'label': 'excellent_with_positive_trend', 'short_reason': 'Strong overall grades with recent improvement in average grade.'}, {'first_name': 'Nir', 'last_name': 'Hazan', 'label'

Merge Memory

In [50]:
# =========================
# Merge all memory files into unified student memory
# =========================
import json
from collections import defaultdict

memory_files = [
    f"Memory/memory_declining_{current_snapshot_date_str}.json",
    f"Memory/memory_low_submission_{current_snapshot_date_str}.json",
    f"Memory/memory_top_students_{current_snapshot_date_str}.json",
    f"Memory/memory_weak_students_{current_snapshot_date_str}.json",
]

all_memory_entries = []

# -------------------------
# Load all memory files
# -------------------------
for file_name in memory_files:
    try:
        with open(file_name, "r", encoding="utf-8") as f:
            data = json.load(f)
            if isinstance(data, list):
                all_memory_entries.extend(data)
            else:
                print(f"⚠️ File {file_name} does not contain a list")
    except FileNotFoundError:
        print(f"⚠️ File not found: {file_name}")
    except Exception as e:
        print(f"⚠️ Failed to load {file_name}: {e}")

print("Total raw memory entries:", len(all_memory_entries))

# -------------------------
# Group by student_id
# -------------------------
student_memory_dict = defaultdict(lambda: {
    "student_id": None,
    "first_name": None,
    "last_name": None,
    "signals": []
})

for entry in all_memory_entries:
    student_id = entry.get("student_id")
    if student_id is None:
        print(f"⚠️ Skipping entry without student_id: {entry}")
        continue

    student_memory_dict[student_id]["student_id"] = int(student_id)
    student_memory_dict[student_id]["first_name"] = entry.get("first_name")
    student_memory_dict[student_id]["last_name"] = entry.get("last_name")

    signal = {
        "signal_type": entry.get("signal_type", "unknown"),
        "label": entry.get("label"),
        "reason": entry.get("reason"),
        "last_updated": entry.get("last_updated")
    }

    student_memory_dict[student_id]["signals"].append(signal)

# -------------------------
# Convert to list
# -------------------------
student_memory = list(student_memory_dict.values())

# Optional: sort
student_memory = sorted(student_memory, key=lambda x: x["student_id"])

# -------------------------
# Save unified memory
# -------------------------
with open(f"Memory/student_memory_{current_snapshot_date_str}.json", "w", encoding="utf-8") as f:
    json.dump(student_memory, f, ensure_ascii=False, indent=2)

print("Unified student memory saved:", len(student_memory))
print(student_memory[:3])

Total raw memory entries: 7
Unified student memory saved: 7
[{'student_id': 9, 'first_name': 'Dan', 'last_name': 'Cohen', 'signals': [{'signal_type': 'low_performance', 'label': 'struggling_academically', 'reason': 'High submission rate with low average grades.', 'last_updated': '2025-11-01'}]}, {'student_id': 10, 'first_name': 'Noa', 'last_name': 'Levy', 'signals': [{'signal_type': 'low_performance', 'label': 'struggling_academically', 'reason': 'High submission rate with low average grades.', 'last_updated': '2025-11-01'}]}, {'student_id': 11, 'first_name': 'Itay', 'last_name': 'Mizrahi', 'signals': [{'signal_type': 'low_performance', 'label': 'struggling_academically', 'reason': 'High submission rate with very low average grades.', 'last_updated': '2025-11-01'}]}]


Find students with multiple signals

In [51]:
# =========================
# Find students with multiple signals
# =========================
import json
import pandas as pd

# -------------------------
# Load unified memory
# -------------------------
with open(f"Memory/student_memory_{current_snapshot_date_str}.json", "r", encoding="utf-8") as f:
    student_memory = json.load(f)

# -------------------------
# Build summary table
# -------------------------
multi_signal_rows = []

for student in student_memory:
    signals = student.get("signals", [])
    signal_types = [s.get("signal_type", "unknown") for s in signals]

    if len(signals) >= 2:
        multi_signal_rows.append({
            "student_id": student.get("student_id"),
            "FirstName": student.get("first_name"),
            "LastName": student.get("last_name"),
            "signal_count": len(signals),
            "signal_types": ", ".join(signal_types)
        })

multi_signal_students = pd.DataFrame(multi_signal_rows)

# -------------------------
# Sort for readability
# -------------------------
if not multi_signal_students.empty:
    multi_signal_students = multi_signal_students.sort_values(
        ["signal_count", "LastName", "FirstName"],
        ascending=[False, True, True]
    ).reset_index(drop=True)

print("Students with multiple signals:", len(multi_signal_students))
print(multi_signal_students)

Students with multiple signals: 0
Empty DataFrame
Columns: []
Index: []


In [52]:
# =========================
# Detailed view for students with multiple signals
# =========================
import json
import pandas as pd

# -------------------------
# Load unified memory
# -------------------------
with open(f"Memory/student_memory_{current_snapshot_date_str}.json", "r", encoding="utf-8") as f:
    student_memory = json.load(f)

# -------------------------
# Build detailed table
# -------------------------
multi_signal_detail_rows = []

for student in student_memory:
    signals = student.get("signals", [])
    
    if len(signals) >= 2:
        for signal in signals:
            multi_signal_detail_rows.append({
                "student_id": student.get("student_id"),
                "FirstName": student.get("first_name"),
                "LastName": student.get("last_name"),
                "signal_count": len(signals),
                "signal_type": signal.get("signal_type"),
                "label": signal.get("label"),
                "reason": signal.get("reason"),
                "last_updated": signal.get("last_updated")
            })

multi_signal_details = pd.DataFrame(multi_signal_detail_rows)

# -------------------------
# Sort for readability
# -------------------------
if not multi_signal_details.empty:
    multi_signal_details = multi_signal_details.sort_values(
        ["signal_count", "LastName", "FirstName", "signal_type"],
        ascending=[False, True, True, True]
    ).reset_index(drop=True)

print("Detailed rows for students with multiple signals:", len(multi_signal_details))
print(multi_signal_details)

Detailed rows for students with multiple signals: 0
Empty DataFrame
Columns: []
Index: []


In [53]:
# =========================
# Save detailed multi-signal report
# =========================
if not multi_signal_details.empty:
    multi_signal_details.to_csv(
        f"ProcessedData/multi_signal_details_{current_snapshot_date_str}.csv",
        index=False
    )
    print("Saved to multi_signal_details file")
else:
    print("No multi-signal students to save")

No multi-signal students to save


In [54]:
# =========================
# One row per student (aggregated view)
# =========================
expected_cols = [
    "student_id",
    "FirstName",
    "LastName",
    "signal_count",
    "signal_type",
    "label",
    "reason",
    "last_updated",
]

if multi_signal_details.empty:
    print("multi_signal_details is empty")
else:
    missing_cols = [col for col in expected_cols if col not in multi_signal_details.columns]
    
    if missing_cols:
        print("Missing columns in multi_signal_details:", missing_cols)
        print("Available columns:", list(multi_signal_details.columns))
    else:
        grouped = multi_signal_details.groupby(
            ["student_id", "FirstName", "LastName"],
            as_index=False
        ).agg({
            "signal_type": lambda x: ", ".join(sorted(set(x))),
            "label": lambda x: ", ".join(sorted(set(x))),
            "reason": lambda x: " | ".join(x.astype(str))
        })

        print(grouped)

multi_signal_details is empty


#### Agentic

In [55]:
# =========================
# Agent action recommendations for all students with multiple signals
# =========================
import json
import re
from llm_client import llm_call

def safe_parse_llm_json(text):
    try:
        return json.loads(text)
    except:
        pass

    try:
        json_match = re.search(r"\{.*\}", text, re.DOTALL)
        if json_match:
            return json.loads(json_match.group())
    except:
        pass

    print("⚠️ Failed to parse LLM output as JSON")
    return None


# -------------------------
# Load unified memory
# -------------------------
with open(f"Memory/student_memory_{current_snapshot_date_str}.json", "r", encoding="utf-8") as f:
    student_memory = json.load(f)

# -------------------------
# Keep only students with multiple signals
# -------------------------
target_students = [
    student for student in student_memory
    if len(student.get("signals", [])) >= 2
]

print("Students with multiple signals:", len(target_students))

# -------------------------
# Shared system prompt
# -------------------------
system_prompt = """
You are an academic support assistant for a homeroom teacher.

You will receive the memory card of one student.
Your task is to recommend the most appropriate next action for the homeroom teacher.

Possible actions:
1. monitor
   Use this when the student should be watched further, but no direct intervention is needed yet.

2. check_in_with_student
   Use this when the teacher should talk to the student soon because the signals look concerning.

3. praise_student
   Use this when the student's signals are positive and the teacher should reinforce progress or strong performance.

Important rules:
- Be cautious.
- Use only the provided signals.
- Do not invent personal, family, or psychological causes.
- Return only valid JSON.
- Include:
  student_id, first_name, last_name, recommended_action, priority, short_reason

Priority must be one of:
- low
- medium
- high
"""

# -------------------------
# Run LLM for each student
# -------------------------
student_action_recommendations = []

for i, student in enumerate(target_students, start=1):
    print(f"\nProcessing {i}/{len(target_students)}: "
          f"{student.get('first_name')} {student.get('last_name')}")

    user_prompt = f"""
Student memory:
{json.dumps(student, ensure_ascii=False, indent=2)}

Return JSON in this exact format:
{{
  "student_id": 25,
  "first_name": "Name",
  "last_name": "Surname",
  "recommended_action": "check_in_with_student",
  "priority": "high",
  "short_reason": "The student has multiple concerning academic signals, including low performance and decline."
}}
"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    raw_output = llm_call(
        messages,
        max_new_tokens=300
    )

    parsed_action = safe_parse_llm_json(raw_output)

    if parsed_action is None:
        print(f"⚠️ Could not parse recommendation for "
              f"{student.get('first_name')} {student.get('last_name')}")
        continue

    student_action_recommendations.append(parsed_action)

# -------------------------
# Save all recommendations
# -------------------------
with open(f"Memory/student_action_recommendations_{current_snapshot_date_str}.json", "w", encoding="utf-8") as f:
    json.dump(student_action_recommendations, f, ensure_ascii=False, indent=2)

print("\nSaved to student_action_recommendations.json")
print("Recommendations:", len(student_action_recommendations))

# -------------------------
# Preview
# -------------------------
print(json.dumps(student_action_recommendations[:5], ensure_ascii=False, indent=2))

Students with multiple signals: 0

Saved to student_action_recommendations.json
Recommendations: 0
[]


#### Merge student memory with action recommendations

In [56]:
# =========================
# Merge student memory with action recommendations
# =========================
import json

# -------------------------
# Load files
# -------------------------
with open(f"Memory/student_memory_{current_snapshot_date_str}.json", "r", encoding="utf-8") as f:
    student_memory = json.load(f)

with open(f"Memory/student_action_recommendations_{current_snapshot_date_str}.json", "r", encoding="utf-8") as f:
    student_action_recommendations = json.load(f)

# -------------------------
# Index actions by student_id
# -------------------------
actions_by_student_id = {}

for action in student_action_recommendations:
    student_id = action.get("student_id")
    if student_id is None:
        continue

    actions_by_student_id[int(student_id)] = {
        "recommended_action": action.get("recommended_action"),
        "priority": action.get("priority"),
        "reason": action.get("short_reason"),
        "date": current_snapshot_date_str
    }

# -------------------------
# Attach actions to student memory
# -------------------------
student_memory_with_actions = []

for student in student_memory:
    student_id = int(student["student_id"])

    student_copy = dict(student)

    if student_id in actions_by_student_id:
        student_copy["actions"] = [actions_by_student_id[student_id]]
    else:
        student_copy["actions"] = []

    student_memory_with_actions.append(student_copy)

# -------------------------
# Save merged file
# -------------------------
with open(f"Memory/student_memory_with_actions_{current_snapshot_date_str}.json", "w", encoding="utf-8") as f:
    json.dump(student_memory_with_actions, f, ensure_ascii=False, indent=2)

print("Saved to Memory/student_memory_with_actions.json")
print("Students in merged memory:", len(student_memory_with_actions))
print(json.dumps(student_memory_with_actions[:3], ensure_ascii=False, indent=2))

Saved to Memory/student_memory_with_actions.json
Students in merged memory: 7
[
  {
    "student_id": 9,
    "first_name": "Dan",
    "last_name": "Cohen",
    "signals": [
      {
        "signal_type": "low_performance",
        "label": "struggling_academically",
        "reason": "High submission rate with low average grades.",
        "last_updated": "2025-11-01"
      }
    ],
    "actions": []
  },
  {
    "student_id": 10,
    "first_name": "Noa",
    "last_name": "Levy",
    "signals": [
      {
        "signal_type": "low_performance",
        "label": "struggling_academically",
        "reason": "High submission rate with low average grades.",
        "last_updated": "2025-11-01"
      }
    ],
    "actions": []
  },
  {
    "student_id": 11,
    "first_name": "Itay",
    "last_name": "Mizrahi",
    "signals": [
      {
        "signal_type": "low_performance",
        "label": "struggling_academically",
        "reason": "High submission rate with very low average grades.",